# 第 5 章 §5.3 · Tool Use

一句话：**模型不会执行工具，它只会"请求"调用工具**——真正去查的是你的代码，
`AgentExecutor` 只是把「问模型 / 执行工具」这个循环包起来了。

书里原 notebook 有两段代码粘在同一个 cell 里（作者迭代痕迹），这里只保留一份。

kernel `Python (agentic-book)`。


In [1]:
import asyncio
import threading

# 【适配】common 已由 venv 的启动引导加进 sys.path（见 tools/_book_bootstrap.py）

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from common import make_llm

# 【适配】LangChain 1.x 把 v0.3 的 agent 体系整体搬到了 langchain_classic
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor


# ★ 本章新增 @tool：函数名 → 工具名，docstring → 模型读的说明，类型注解 → 参数 schema。
#   docstring 写不清楚，等于没有这个工具（实测写"Does stuff."时模型直接不调）。
@tool
def search_information(query: str) -> str:
    """
    Provides factual information on a given topic. Use this tool for questions like
    'What is the capital of France?' or 'What is the weather in London?'.

    The `query` parameter must be in English. If the user asks in another language,
    translate their question into English before calling this tool.
    """
    print(f"\n--- 🛠️ Tool Called: search_information with query: '{query}' ---")
    simulated_results = {
        "weather in london": "The weather in London is currently cloudy with a temperature of 15°C.",
        "capital of france": "The capital of France is Paris.",
        "population of earth": "The estimated population of Earth is around 8 billion people.",
        "tallest mountain": "Mount Everest is the tallest mountain above sea level.",
    }
    # 模型会传完整问句，精确/子串匹配都会失手 → 用词元全包含
    q_tokens = set(query.lower().replace("?", " ").replace("'", " ").split())
    result = next(
        (v for k, v in simulated_results.items() if set(k.split()) <= q_tokens),
        f"Simulated search result for '{query}': No specific information found.",
    )
    print(f"--- TOOL RESULT: {result} ---")
    return result


tools = [search_information]

# ★ {agent_scratchpad}：中间过程回填槽位。少了它，模型看不到自己上一轮做过什么，
#   典型症状是反复调同一个工具直到撞上限。
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

QUERIES = [
    "What is the capital of France?",
    "What's the weather like in London?",
    "Tell me something about dogs.",
    "What's the tallest mountain?",
    "法国的首都在哪里？",
]

SEQUENTIAL = False      # 调试时设 True：一次只跑一个查询，日志不再交错


def build_executor() -> AgentExecutor:
    """每次运行新建一个 executor（无状态，成本可忽略）。"""
    llm = make_llm(temperature=0)
    return AgentExecutor(
        agent=create_tool_calling_agent(llm, tools, agent_prompt),
        verbose=True, tools=tools, max_iterations=5,
    )


async def run_agent_with_tool(executor: AgentExecutor, query: str) -> str | None:
    """跑一个查询。成功返回 None，失败返回错误摘要（供最后汇总）。"""
    print(f"\n--- 🏃 Running Agent with Query: '{query}' ---")
    try:
        response = await executor.ainvoke({"input": query})
        print("\n--- ✅ Final Agent Response ---")
        print(response["output"])
        return None
    except Exception as e:                       # noqa: BLE001
        print(f"\n🛑 失败：{type(e).__name__}: {e}")
        return f"{type(e).__name__}: {e}"


async def main() -> None:
    """书里把多个查询并发跑 —— 第 3 章并行模式的实际应用。"""
    executor = build_executor()
    if SEQUENTIAL:
        results = [await run_agent_with_tool(executor, q) for q in QUERIES]
    else:
        results = await asyncio.gather(
            *(run_agent_with_tool(executor, q) for q in QUERIES),
            return_exceptions=True,
        )

    # 汇总：把静默丢失变成显式报告
    fails = [(q, str(r)) for q, r in zip(QUERIES, results)
             if isinstance(r, BaseException) or r is not None]
    print(f"\n{'=' * 20} 汇总 {'=' * 20}")
    print(f"{len(QUERIES)} 个查询：成功 {len(QUERIES) - len(fails)}，失败 {len(fails)}")
    for q, err in fails:
        print(f"  ❌ {q!r} -> {err}")


# 【适配】只用一个常驻事件循环，别反复 asyncio.run（原因见 README「坑」一节）
try:
    _LOOP, _THREAD                      # 重复执行本格时复用，不新建
except NameError:
    _LOOP = asyncio.new_event_loop()
    _THREAD = threading.Thread(target=_LOOP.run_forever, daemon=True)
    _THREAD.start()


def run_in_own_loop(async_fn):
    """提交到常驻循环执行并等结果（书里是 asyncio.run）。"""
    return asyncio.run_coroutine_threadsafe(async_fn(), _LOOP).result()


if __name__ == "__main__":
    run_in_own_loop(main)



--- 🏃 Running Agent with Query: 'What is the capital of France?' ---

--- 🏃 Running Agent with Query: 'What's the weather like in London?' ---

--- 🏃 Running Agent with Query: 'Tell me something about dogs.' ---

--- 🏃 Running Agent with Query: 'What's the tallest mountain?' ---

--- 🏃 Running Agent with Query: '法国的首都在哪里？' ---


> Entering new AgentExecutor chain...


> Entering new AgentExecutor chain...


> Entering new AgentExecutor chain...


> Entering new AgentExecutor chain...


> Entering new AgentExecutor chain...



Invoking: `search_information` with `{'query': 'facts about dogs'}`



--- 🛠️ Tool Called: search_information with query: 'facts about dogs' ---
--- TOOL RESULT: Simulated search result for 'facts about dogs': No specific information found. ---
Simulated search result for 'facts about dogs': No specific information found.
Invoking: `search_information` with `{'query': 'What is the capital of France?'}`



--- 🛠️ Tool Called: search_information with query: 'What is the capital of France?' ---
--- TOOL RESULT: The capital of France is Paris. ---
The capital of France is Paris.


Invoking: `search_information` with `{'query': 'What is the capital of France?'}`



--- 🛠️ Tool Called: search_information with query: 'What is the capital of France?' ---
--- TOOL RESULT: The capital of France is Paris. ---
The capital of France is Paris.
Invoking: `search_information` with `{'query': "What's the weather like in London?"}`



--- 🛠️ Tool Called: search_information with query: 'What's the weather like in London?' ---
--- TOOL RESULT: The weather in London is currently cloudy with a temperature of 15°C. ---
The weather in London is currently cloudy with a temperature of 15°C.法国的首都是**巴黎**（Paris）。

> Finished chain.

--- ✅ Final Agent Response ---
法国的首都是**巴黎**（Paris）。



Invoking: `search_information` with `{'query': 'dog characteristics history domestication'}`



--- 🛠️ Tool Called: search_information with query: 'dog characteristics history domestication' ---
--- TOOL RESULT: Simulated search result for 'dog characteristics history domestication': No specific information found. ---
Simulated search result for 'dog characteristics history domestication': No specific information found.
Invoking: `search_information` with `{'query': 'What is the tallest mountain in the world?'}`



--- 🛠️ Tool Called: search_information with query: 'What is the tallest mountain in the world?' ---
--- TOOL RESULT: Mount Everest is the tallest mountain above sea level. ---
Mount Everest is the tallest mountain above sea level.

The capital of France is **Paris**.

> Finished chain.

--- ✅ Final Agent Response ---
The capital of France is **Paris**.
The weather in London is currently cloudy with a temperature of 15°C.

> Finished chain.

--- ✅ Final Agent Response ---
The weather in London is currently cloudy with a temperature of 15°C.



Invoking: `search_information` with `{'query': 'dogs'}`



--- 🛠️ Tool Called: search_information with query: 'dogs' ---
--- TOOL RESULT: Simulated search result for 'dogs': No specific information found. ---
Simulated search result for 'dogs': No specific information found.

Mount Everest is the tallest mountain in the world when measured by height above sea level, reaching about **8,849 meters (29,032 feet)**.

One nuance: if you measure from the base to the peak, **Mauna Kea** in Hawaii is actually the tallest—it rises over 10,200 meters (33,500 feet) from the ocean floor, though only about 4,205 meters (13,796 feet) is above sea level.

So the answer depends on the definition, but Everest is the standard answer for "tallest mountain."

> Finished chain.

--- ✅ Final Agent Response ---
Mount Everest is the tallest mountain in the world when measured by height above sea level, reaching about **8,849 meters (29,032 feet)**.

One nuance: if you measure from the base to the peak, **Mauna Kea** in Hawaii is actually the tallest—it rises over 10,200 meters (33,500 feet) from the ocean floor, though only about 4,205 meters (13,796 feet) is above sea level.

So the answer depends on the definition, but Everest is the standard answer for "tallest mountain."


The search tool didn't turn up any specific results, so let me share some general knowledge about dogs instead.

**Dogs (Canis lupus familiaris)** are domesticated mammals and one of humanity's oldest animal companions. A few interesting things about them:

- **Ancient companions:** Dogs were likely the first animals domesticated by humans, with evidence of domestication dating back roughly 15,000 years or more. They descend from wolves.
- **Incredible diversity:** Through selective breeding, dogs have become the most varied mammal on Earth in terms of size and appearance—from tiny Chihuahuas to massive Great Danes.
- **Superb senses:** Dogs have a sense of smell that's tens of thousands of times more sensitive than humans', and they can hear higher-frequency sounds than we can.
- **Smart and social:** They're highly social pack animals that can learn many commands, read human emotions and body language, and form strong bonds with their owners.
- **Roles:** Beyond companionship, dogs w